# BDDK Taşıt Kredisi Heuristik Ön-Elemesi — Ders Kitabı Notebooku

Bu notebook Model 12'nin yeniden üretilebilir denetim yüzeyidir. İn-sample/permutasyon taraması bir OOF performans ölçümü değildir; yalnız cari/revize BDDK serisinin dört ön-kayıtlı dönüşümünün ilk-yayım vintajı edinme maliyetini gerekçelendirip gerekçelendirmediğini inceler.

## Okuma hedefleri

- M−2 bilgi kesiminin ve konumsal hafta indekslemesinin denetlenmesi.
- Tatil nedeniyle kayan BDDK referans haftalarının veri bozulmasından ayrılması.
- Model 11 kontrol harness'inin yeniden üretildiğinin görülmesi.
- Doygun RF/HGB sonuçlarını karar kapısından dışarıda tutarak iki lojistik kolun karşılaştırılması.
- `tarama_kesinligi=HEURISTIK` yorum sınırının korunması.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

repo = Path.cwd()
if not (repo / 'data').exists():
    repo = repo.parent
model_dir = repo / 'data' / 'processed' / 'model'
ozet = json.loads((model_dir / 'model_12_bddk_tavan_ozet.json').read_text(encoding='utf-8'))
seri = pd.read_csv(model_dir / 'model_12_bddk_tasit_haftalik_cari.csv')
feature = pd.read_csv(model_dir / 'model_12_bddk_aylik_features.csv')
print(f"Model: {ozet['model']} | test: {ozet['test']} | kesinlik: {ozet['karar']['tarama_kesinligi']}")

Model: Model 12 BDDK heuristik on-eleme | test: 2025-07..2026-06 ACILMADI_KILITLI | kesinlik: HEURISTIK


## Bilgi zamanı ve dört sabit dönüşüm

Origin ayı `M` için `w0`, `M−2` ay sonuna eşit veya ondan önceki son yayımlanmış BDDK referans haftasıdır. `w0−k` takvimden gün çıkarmak değil, yayımlanmış seride `k` gözlem geri gitmektir. Dört özellik: 4, 13 ve 52 gözlemlik nominal bakiye değişimleri ile `((1+n/100)/(1+p/100)−1)×100` bileşik reel 4-hafta değişimidir. `p`, mevcut hatta yüzdelik puan olarak saklanan M−2 aylık TÜFE değişimidir.

In [2]:
takvim = ozet['bddk_serisi']['takvim_denetimi']
display(pd.DataFrame([{
    'gözlem': takvim['gozlem_sayisi'],
    'ilk hafta': ozet['bddk_serisi']['ilk_hafta'],
    'son hafta': ozet['bddk_serisi']['son_hafta'],
    'min/maks ardışık gün': f"{takvim['ardisik_aralik_min_gun']}/{takvim['ardisik_aralik_maks_gun']}",
    'Cuma dışı': takvim['cuma_disi_hafta_sayisi'],
    'eşleşmeyen tatil kayması': len(takvim['tatille_eslesmeyen_haftalar']),
}]))
display(pd.DataFrame(takvim['tatille_eslesen_haftalar']).head(8))
display(pd.DataFrame([ozet['feature_denetimi']['aralik_sapma_7gun_ustu']]))
display(feature.head(3))
display(feature.tail(3))

,gözlem,ilk hafta,son hafta,min/maks ardışık gün,Cuma dışı,eşleşmeyen tatil kayması
0,657,2014-01-03,2026-07-31,3/11,26,0


,referans_hafta,tatil
0,2015-04-30,1 Mayıs Emek ve Dayanışma Günü
1,2015-07-16,Ramazan Bayramı
2,2015-09-23,Kurban Bayramı
3,2015-12-31,Yılbaşı
4,2017-05-18,"19 Mayıs Atatürk'ü Anma, Gençlik ve Spor Bayramı"
5,2017-08-31,Kurban Bayramı
6,2018-06-14,Ramazan Bayramı
7,2018-08-20,Kurban Bayramı


,4h,13h,52h
0,0,0,0


,hedef_ay,bddk_capa_haftasi,bddk_m2_son_gun,bddk_tasit_bakiye_4h_degisim_pct,bddk_tasit_bakiye_13h_degisim_pct,bddk_tasit_bakiye_52h_degisim_pct,bddk_tasit_bakiye_reel_4h_degisim_pct,gerceklesen_aralik_4h_gun,gerceklesen_aralik_13h_gun,gerceklesen_aralik_52h_gun,aralik_sapma_4h_7gun_ustu,aralik_sapma_13h_7gun_ustu,aralik_sapma_52h_7gun_ustu,asimetrik_tatil_kesimi_4h,asimetrik_tatil_kesimi_13h,asimetrik_tatil_kesimi_52h
0,2021-03,2021-01-29,2021-01-31,1.291943,6.813996,71.065658,-0.383429,29,91,364,False,False,False,False,False,False
1,2021-04,2021-02-26,2021-02-28,3.790224,10.407842,76.663129,2.856441,28,91,364,False,False,False,False,False,False
2,2021-05,2021-03-26,2021-03-31,7.692981,15.178257,88.715622,6.547201,28,91,364,False,False,False,False,False,False


,hedef_ay,bddk_capa_haftasi,bddk_m2_son_gun,bddk_tasit_bakiye_4h_degisim_pct,bddk_tasit_bakiye_13h_degisim_pct,bddk_tasit_bakiye_52h_degisim_pct,bddk_tasit_bakiye_reel_4h_degisim_pct,gerceklesen_aralik_4h_gun,gerceklesen_aralik_13h_gun,gerceklesen_aralik_52h_gun,aralik_sapma_4h_7gun_ustu,aralik_sapma_13h_7gun_ustu,aralik_sapma_52h_7gun_ustu,asimetrik_tatil_kesimi_4h,asimetrik_tatil_kesimi_13h,asimetrik_tatil_kesimi_52h
47,2025-02,2024-12-27,2024-12-31,0.470963,-4.251954,-21.123388,-0.551505,28,91,364,False,False,False,False,False,False
48,2025-03,2025-01-31,2025-01-31,-4.401100,-5.799147,-24.261280,-8.981601,28,91,364,False,False,False,False,False,False
49,2025-04,2025-02-28,2025-02-28,-3.914036,-9.018924,-26.779139,-6.049835,28,91,364,False,False,False,False,False,False


## Kontrol harness'i

Kol 1 yalnız Model 11 feature setini kullanır. Gözlenen MCC ve null95 değerleri ön-kayıtlı referans toleranslarını aşarsa BDDK'lı kol yorumlanmaz. Aynı 50 aylık etiket dizisi, seed ve permütasyon matrisi iki kolda ortaktır.

In [3]:
harness = pd.DataFrame(ozet['harness']).T
harness['gecti'] = harness['gecti'].astype(bool)
display(harness[['en_buyuk_mutlak_fark', 'gecti', 'not_dusuldu']])
assert harness['gecti'].all(), 'Kontrol harness geçmedi; test kolu yorumlanamaz.'

,en_buyuk_mutlak_fark,gecti,not_dusuldu
lojistik_l2_c01,0.0,True,False
lojistik_l2_c1,0.0,True,False
random_forest_sigin,0.0,True,False
hist_gradient_sigin,0.0,True,False


## İki kollu ön-eleme kapısı

`marj = gözlenen − null95`; `delta_marj = marj_kol2 − marj_kol1`. Yalnız iki lojistik konfigürasyon karar verir. Kol 2 marjı en az `+0,15` ise `ON_ELEME_GECTI`; bu yokken delta marj en az `+0,15` ise `ON_ELEME_ZAYIF`; aksi durumda `ON_ELEME_ISARET_YOK`. RF ve HGB yalnız doygunluk denetimidir.

In [4]:
satirlar = []
for ad, k1 in ozet['kol1_kontrol'].items():
    k2 = ozet['kol2_bddk_ekli'][ad]
    k = ozet['karar']['karsilastirma'][ad]
    satirlar.append({
        'konfigürasyon': ad,
        'kol1 gözlenen': k1['tavan_gozlenen'],
        'kol1 null95': k1['tavan_null95'],
        'kol1 marj': k1['marj'],
        'kol2 gözlenen': k2['tavan_gozlenen'],
        'kol2 null95': k2['tavan_null95'],
        'kol2 marj': k2['marj'],
        'delta marj': k['delta_marj'],
        'doygun': k['doygun'],
    })
karsilastirma = pd.DataFrame(satirlar).set_index('konfigürasyon')
display(karsilastirma.style.format(precision=4))
karar = ozet['karar']
display(Markdown(f"**Hüküm:** `{karar['hukum']}` — **kesinlik:** `{karar['tarama_kesinligi']}` — **otomatik dal:** `{karar['otomatik_sonraki_dal']}`"))

,kol1 gözlenen,kol1 null95,kol1 marj,kol2 gözlenen,kol2 null95,kol2 marj,delta marj,doygun
konfigürasyon,,,,,,,,
lojistik_l2_c01,0.2148,0.4450,-0.2303,0.3794,0.5006,-0.1211,0.1091,False
lojistik_l2_c1,0.1690,0.4684,-0.2994,0.4936,0.5528,-0.0592,0.2402,False
random_forest_sigin,0.9169,0.9155,0.0013,0.9690,0.9537,0.0153,0.0140,True
hist_gradient_sigin,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000,0.0000,True


**Hüküm:** `ON_ELEME_ZAYIF` — **kesinlik:** `HEURISTIK` — **otomatik dal:** `KAPASITE_DUSURULMUS_TEKRAR`

## Yorum sınırı

Pozitif sonuç yalnız ilk-yayım vintajı edinme maliyetini gerekçelendirir; terfi veya üretim becerisi değildir. Negatif sonuç BDDK'nın sinyalsiz olduğunu, temiz vintajın başarısız olacağını ya da başka dönüşümlerin işe yaramayacağını kanıtlamaz. Cari/revize seri için kalem düzeyinde revizyon sınırı belgelenmediğinden bütün hükümler `HEURISTIK` etiketiyle okunur. Kilitli `2025-07..2026-06` testi bu notebookta açılmaz.